In [8]:
from typing import TypedDict

from langgraph.constants import START,END
from langgraph.graph import StateGraph
from model import qwen

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
class AgentState(TypedDict):
    summary: str
    entities: str
    search_result: str

In [ ]:
graph = StateGraph(AgentState)
llm = qwen

In [ ]:
# ======================
# 3 个并行节点
# ======================
def summary_action(state:AgentState):
    summary = AgentState['summary']
    return f"搜索结果：{summary} 相关资料"

def extract_action(state:AgentState):
    entities = llm.invoke(f"提取实体：{state['input']}").content
    return {"entities": entities}

def search_action(state:AgentState):
    res = llm.invoke(state["input"])
    return {"search_result": res}

In [ ]:
# ======================
# 最终汇总节点
# ======================
def generate_action(state: AgentState):
    report = f"""
    总结：{state['summary']}
    实体：{state['entities']}
    搜索：{state['search_result']}
    """
    final = llm.invoke(f"生成最终报告：{report}").content
    return {"report": final}


In [ ]:
# 🔥 并行执行 3 个节点
graph.add_node("parallel_node", [summary_action, extract_action, search_action])

graph.add_node("generate_node", generate_action)

# 流程
graph.set_entry_point(START)
graph.set_entry_point(END)
graph.add_edge(START, "parallel_node")
graph.add_edge("parallel_node", "generate_node")
graph.add_edge("generate_node", END)

app = graph.compile()

In [2]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))


NameError: name 'app' is not defined